In [1]:
from pyspark.sql.functions import col, current_timestamp, lit
from delta.tables import DeltaTable

# ----------------------------------------------------------------------
# Step 1: Drop any old tables (clean slate for demo)
# ----------------------------------------------------------------------
spark.sql("DROP TABLE IF EXISTS main_dim")
spark.sql("DROP TABLE IF EXISTS staging_changes")
print("Old tables dropped (clean slate)")

# ----------------------------------------------------------------------
# Step 2: Create initial main Delta table (SCD Type 2 dimension)
# ----------------------------------------------------------------------
initial_data = spark.createDataFrame([
    (1, "Alice", 50000.0),
    (2, "Bob", 60000.0)
], ["id", "name", "salary"])

# Add SCD Type 2 columns
initial_data = initial_data.withColumn("valid_from", current_timestamp()) \
                           .withColumn("valid_to", lit(None).cast("timestamp")) \
                           .withColumn("is_current", lit(True))

initial_data.write.format("delta").mode("overwrite").saveAsTable("main_dim")

print("\n=== BEFORE MERGE: Initial main_dim state ===")
spark.table("main_dim").show(truncate=False)

# ----------------------------------------------------------------------
# Step 3: Simulate new incoming changes (staging table)
# ----------------------------------------------------------------------
# In real life: this would be your new CSV / incremental data
new_changes = spark.createDataFrame([
    # Update: Alice got a raise (same id, new salary)
    (1, "Alice", 55000.0),
    # Insert: completely new employee
    (3, "Charlie", 70000.0)
], ["id", "name", "salary"])

new_changes = new_changes.withColumn("valid_from", current_timestamp()) \
                         .withColumn("valid_to", lit(None).cast("timestamp")) \
                         .withColumn("is_current", lit(True))

new_changes.write.format("delta").mode("overwrite").saveAsTable("staging_changes")

print("\n=== New incoming changes (staging) ===")
spark.table("staging_changes").show(truncate=False)

# ----------------------------------------------------------------------
# Step 4: Apply SCD Type 2 MERGE
# ----------------------------------------------------------------------
delta_main = DeltaTable.forName(spark, "main_dim")

delta_main.alias("main").merge(
    spark.table("staging_changes").alias("new"),
    "main.id = new.id AND main.is_current = true"
).whenMatchedUpdate(set={
    "valid_to": current_timestamp(),
    "is_current": lit(False)
}).whenNotMatchedInsertAll().execute()

print("\nSCD Type 2 MERGE completed (history preserved)")

# ----------------------------------------------------------------------
# Step 5: Print AFTER MERGE (full history)
# ----------------------------------------------------------------------
print("\n=== AFTER MERGE: main_dim with history ===")
spark.table("main_dim").show(truncate=False)

# Show only current records
print("\n=== Current active records (is_current = true) ===")
spark.table("main_dim").filter(col("is_current") == True).show(truncate=False)

# Show history for a specific id (Alice was updated)
print("\n=== History for id = 1 (Alice) ===")
spark.table("main_dim") \
    .filter(col("id") == 1) \
    .select("id", "name", "salary", "valid_from", "valid_to", "is_current") \
    .orderBy("valid_from") \
    .show(truncate=False)

# Step 6: Cleanup (optional - run to reset for next test)
# spark.sql("DROP TABLE IF EXISTS main_dim")
# spark.sql("DROP TABLE IF EXISTS staging_changes")
# print("Tables cleaned up!")

StatementMeta(, 8733f30a-a388-46ba-80d2-6798793431e8, 3, Finished, Available, Finished)

Old tables dropped (clean slate)

=== BEFORE MERGE: Initial main_dim state ===
+---+-----+-------+--------------------------+--------+----------+
|id |name |salary |valid_from                |valid_to|is_current|
+---+-----+-------+--------------------------+--------+----------+
|1  |Alice|50000.0|2026-02-01 12:57:47.476353|NULL    |true      |
|2  |Bob  |60000.0|2026-02-01 12:57:47.476353|NULL    |true      |
+---+-----+-------+--------------------------+--------+----------+


=== New incoming changes (staging) ===
+---+-------+-------+--------------------------+--------+----------+
|id |name   |salary |valid_from                |valid_to|is_current|
+---+-------+-------+--------------------------+--------+----------+
|3  |Charlie|70000.0|2026-02-01 12:58:01.653037|NULL    |true      |
|1  |Alice  |55000.0|2026-02-01 12:58:01.653037|NULL    |true      |
+---+-------+-------+--------------------------+--------+----------+


SCD Type 2 MERGE completed (history preserved)

=== AFTER MERG